In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../Datasets/processed/UHPC_dataset/initial_cleaned.csv", index_col=0)
df = df.reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (2072, 43)


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,fiber2_tensile_strength,fiber2_youngs_modulus,water,sp_type,sp_amount,curing_method,curing_temp,curing_humidity,curing_pressure,cs_28d
0,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,Polycarboxylate-based HRWRA,59.33,Standard Curing,NaN,NaN,NaN,132.0
1,839.0,Type I/II low-alkali portland cement,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,Polycarboxylate-based HRWRA,59.33,Standard Curing,NaN,NaN,NaN,122.5
2,839.0,Type I/II low-alkali portland cement,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,Polycarboxylate-based HRWRA,62.15,Standard Curing,NaN,NaN,NaN,116.0
3,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,Polycarboxylate-based HRWRA,64.98,Standard Curing,NaN,NaN,NaN,134.0
4,839.0,Type I/II low-alkali portland cement,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,NaN,147.0,Polycarboxylate-based HRWRA,64.98,Standard Curing,NaN,NaN,NaN,131.5


# Semantic Missingness Analysis

Analyzing and recoding missing values in UHPC dataset, distinguishing between structural zeros (not applicable) and actual missing data (unknown).

In [2]:
## Data Filtering Utility

def drop_by_threshold(df, threshold_pct):
    """Drop columns where missing values exceed threshold percentage."""
    threshold = threshold_pct * len(df)
    dropped = df.columns[df.isna().sum() > threshold].tolist()
    print(f"Dropped columns ({int(threshold_pct*100)}% threshold): {dropped}")
    return df.loc[:, df.isna().sum() <= threshold]

In [3]:
# 50% missing data threshold
df_raw_50 = df.copy()
df_raw_50 = drop_by_threshold(df_raw_50, 0.5)
df_raw_50 = df_raw_50.reset_index(drop=True)
print(f"50% threshold shape: {df_raw_50.shape}")

Dropped columns (50% threshold): ['fly_ash_type', 'slag_type', 'filler_type', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_type', 'fiber2_amount', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
50% threshold shape: (2072, 30)


## Create Datasets with Different Missing Data Thresholds

In [4]:
# 20% missing data threshold
df_raw_20 = df.copy()
df_raw_20 = drop_by_threshold(df_raw_20, 0.2)
df_raw_20 = df_raw_20.reset_index(drop=True)
print(f"20% threshold shape: {df_raw_20.shape}")

Dropped columns (20% threshold): ['cement_grade', 'fly_ash_type', 'slag_type', 'filler_type', 'fiber1_type', 'fiber1_amount', 'fiber1_length', 'fiber1_diameter', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_type', 'fiber2_amount', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
20% threshold shape: (2072, 25)


In [5]:
## Semantic Missingness Recoding

# Define material amount/type pairs for semantic analysis
amount_type_twins = [
    ['fly_ash', 'fly_ash_type'],
    ['slag', 'slag_type'],
    ['filler', 'filler_type'],
    ['sand', 'sand_type'],
    ['fiber1_amount', 'fiber1_type'],
    ['fiber2_amount', 'fiber2_type'],
    ['sp_amount', 'sp_type']
]

In [6]:
def recode_semantic_missingness(df, amount_col, type_col):
    """Recode missing values based on material amount.
    
    Rules:
    - amount > 0 & type NaN → 'unknown_type' (material used, type unknown)
    - amount NaN/0 & type NaN → 'not_applicable' (material not used)
    """
    # Material used but type unknown
    mask1 = (df[amount_col] > 0) & df[type_col].isna()
    df.loc[mask1, type_col] = 'unknown_type'

    # Material not used (amount 0 or NaN) and type NaN
    mask2 = (df[amount_col].isna() | (df[amount_col] == 0)) & df[type_col].isna()
    df.loc[mask2, type_col] = 'not_applicable'
    df.loc[mask2, amount_col] = 0

    return df

# Apply semantic recoding
df_semantic_recoded = df.copy()

for amount_col, type_col in amount_type_twins:
    before_amount = df[amount_col].isna().sum()
    before_type = df[type_col].isna().sum()
    
    recode_semantic_missingness(df_semantic_recoded, amount_col, type_col)
    
    print(f"\n{amount_col} / {type_col}")
    print(f"  NaN amount:      {before_amount} → {df[amount_col].isna().sum()}  (true reporting gap)")
    print(f"  NaN type:        {before_type} → {df[type_col].isna().sum()}  (true reporting gap)")
    print(f"  unknown_type:    {(df[type_col] == 'unknown_type').sum()}  (recoded)")
    print(f"  not_applicable:  {(df[type_col] == 'not_applicable').sum()}  (recoded)")


fly_ash / fly_ash_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1648 → 1648  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

slag / slag_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1980 → 1980  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

filler / filler_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1712 → 1712  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

sand / sand_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        81 → 81  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber1_amount / fiber1_type
  NaN amount:      669 → 669  (true reporting gap)
  NaN type:        669 → 669  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber2_amount / fiber2_type
  NaN amount:      1954 → 1954  (true repo

In [ ]:
import re

recode_semantic_missingness(df_semantic_recoded, 'cement', 'cement_type')

cement_regex_map = [
    (r'high.?sulfate|type.?hs',     'HS_cement'),
    (r'type.?iii|type.?3\b',        'OPC_III'),
    (r'white',                       'white_cement'),
    (r'cem.?ii|cem2|cemii',          'CEM_II'),
    (r'blast.?furnace',              'BFS_cement'),
    (r'pozzolan',                    'pozzolan_cement'),
    (r'ggbs',                        'OPC_I_GGBS'),
    (r'53.?grade|grade.?53|\b53\b',  'OPC_53'),
    (r'52[.,]?5',                    'OPC_52.5'),
    (r'42[.,]?5',                    'OPC_42.5'),
]

df_semantic_recoded['cement_type'] = df_semantic_recoded['cement_type'].apply(
    lambda val: val if (pd.isna(val) or val in ('unknown_type', 'not_applicable'))
    else next((label for pat, label in cement_regex_map if re.search(pat, val.lower())), 'OPC_unknown')
)

print(df_semantic_recoded['cement_type'].value_counts(dropna=False))

In [ ]:
sp_regex_map = [
    (r'poly.{0,3}carb|carboxylate|carboxylic|pce\b|glenium|viscocrete|viscorete|cx-8|vc-2044|he.?200|advacast', 'PCE'),
    (r'naphthalene|snf',  'SNF'),
    (r'acrylic',          'acrylic_SP'),
]

df_semantic_recoded['sp_type'] = df_semantic_recoded['sp_type'].apply(
    lambda val: val if (pd.isna(val) or val in ('unknown_type', 'not_applicable'))
    else next((label for pat, label in sp_regex_map if re.search(pat, val.lower())), 'SP_unknown')
)

print(df_semantic_recoded['sp_type'].value_counts(dropna=False))

In [7]:
# Recoded + 50% missing data threshold
df_semantic_recod_50 = df_semantic_recoded.copy()
df_semantic_recod_50 = drop_by_threshold(df_semantic_recod_50, 0.5)
print(f"Recoded + 50% threshold shape: {df_semantic_recod_50.shape}")

Dropped columns (50% threshold): ['fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
Recoded + 50% threshold shape: (2072, 35)


## Create Final Datasets (Recoded + Threshold)

In [ ]:
# Recoded + 20% missing data threshold
df_semantic_recod_20 = df_semantic_recoded.copy()
df_semantic_recod_20 = drop_by_threshold(df_semantic_recod_20, 0.2)
print(f"Recoded + 20% threshold shape: {df_semantic_recod_20.shape}")


Dropped columns (20% threshold): ['cement_grade', 'fiber1_length', 'fiber1_diameter', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
Recoded + 20% threshold shape: (2072, 32)


In [9]:
df_semantic_recod_50.to_csv("../Datasets/processed//uhpc_dataset/semantic_recoding_features.csv")

In [15]:
df_semantic_recod_20.isnull().sum()[df_semantic_recod_20.isnull().sum() > 0]

cement_type       19
sand_max_size    276
sp_amount          8
curing_temp      157
dtype: int64

In [17]:
df_semantic_recod_20['sand'][df_semantic_recod_20['sand_max_size'].isna()].describe()

count     276.000000
mean      816.853913
std       397.325703
min         0.000000
25%       653.000000
50%       973.350000
75%      1075.000000
max      1756.480000
Name: sand, dtype: float64